# 04 — Betting Backtest: Kelly, Sharpe, and the Finance Layer

Turn the pre-game win-probability model into a **deployable alpha strategy**. For every game the model disagrees with the market, place a bet sized by one of five staking rules and track the bankroll over the historical odds we have on file.

This notebook is positioned as the quant-finance writeup of the project:

- Predictions are **walk-forward** — each test season was unseen at training time. No look-ahead.
- Kelly sizing across **concurrent** NBA games is normalized so Σfᵢ ≤ 50 % of bankroll (the simultaneous-Kelly correction — textbook Kelly assumes sequential bets with known outcomes, but six games tip off in one 7–9 PM window).
- All metrics use the **sparse bet-day equity series**, never calendar-day returns with zero-filled off-season gaps.
- Small-sample honesty: a single season filtered by edge leaves 100–300 bets, so ROI and Sharpe are reported with **bootstrapped 95 % CIs**.

Finance parallels called out along the way:

| Sports betting concept        | Quant finance analog                                    |
|-------------------------------|---------------------------------------------------------|
| `model_prob - devig(market)`  | alpha (signal − consensus)                              |
| Kelly fraction                | log-utility-optimal position size                       |
| Fractional Kelly              | shrinkage estimator for parameter uncertainty           |
| Vig                           | bid-ask spread / transaction cost                       |
| Walk-forward CV               | expanding-window factor backtest (PIT data discipline)  |
| Simultaneous Kelly cap        | gross-exposure limit                                    |
| Drawdown / Calmar             | hedge-fund risk metric                                  |

## 1. Setup + data availability check

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import text

from db import engine
from src.features.matchup import load_team_games
from src.finance.backtest import BacktestConfig, run_backtest
from src.finance.metrics import summary as metrics_summary
from src.finance.staking import (
    FlatStake, FixedFractional, FullKelly, FractionalKelly, ThresholdKelly,
)
from src.models.walkforward import make_xgb_predict_fn, walk_forward_predictions
from src.models.xgboost_model import select_features

sns.set_theme(style='whitegrid', context='notebook', palette='colorblind')
FEATURES_PATH = Path.cwd().parent / 'data' / 'processed' / 'features.parquet'

with engine.connect() as conn:
    n_odds = conn.execute(text('SELECT COUNT(*) FROM odds_snapshots')).scalar()
print(f'odds_snapshots rows: {n_odds}')
if n_odds == 0:
    print(
        '\\nNo historical odds in the DB. Import a Kaggle NBA closing-line CSV:\\n\\n'
        '  python -m src.ingest.ingest_historical_odds --inspect data/raw/<file>.csv\\n'
        '  # then write a JSON column map and:\\n'
        '  python -m src.ingest.ingest_historical_odds --csv data/raw/<file>.csv --column-map data/raw/col_map.json\\n\\n'
        'The rest of this notebook assumes that import has been done.'
    )

odds_snapshots rows: 0
\nNo historical odds in the DB. Import a Kaggle NBA closing-line CSV:\n\n  python -m src.ingest.ingest_historical_odds --inspect data/raw/<file>.csv\n  # then write a JSON column map and:\n  python -m src.ingest.ingest_historical_odds --csv data/raw/<file>.csv --column-map data/raw/col_map.json\n\nThe rest of this notebook assumes that import has been done.


## 2. Walk-forward predictions

Every test row in the frame below comes from a fold where the model was trained *only* on prior seasons. `predicted_at` is set to the last training-game date — a conservative timestamp — and the backtest asserts `predicted_at <= game_date` in its loop.

In [2]:
features_df = pd.read_parquet(FEATURES_PATH)
feature_cols = select_features(features_df)
print(f'{len(features_df):,} games across {features_df["season"].nunique()} seasons, {len(feature_cols)} features')

preds = walk_forward_predictions(
    features_df,
    make_xgb_predict_fn(feature_cols),
    initial_train_seasons=3, mode='expanding',
)
print(f'Walk-forward predictions: {len(preds):,} rows, {preds["fold"].nunique()} folds')
preds.head()

10,569 games across 10 seasons, 70 features
Walk-forward predictions: 7,299 rows, 7 folds


,game_id,game_date,home_team_id,away_team_id,home_won,model_prob,fold,predicted_at
0,0021900133,2019-11-10,1610612753,1610612754,0,0.528741,2019-20,2019-04-10
1,0021900135,2019-11-10,1610612760,1610612749,0,0.418759,2019-20,2019-04-10
2,0021900140,2019-11-11,1610612765,1610612750,0,0.532205,2019-20,2019-04-10
3,0021900142,2019-11-11,1610612740,1610612745,0,0.305226,2019-20,2019-04-10
4,0021900143,2019-11-11,1610612759,1610612763,0,0.805672,2019-20,2019-04-10


## 3. Load odds + outcomes

In [3]:
odds = pd.read_sql_query(
    text('SELECT game_date, home_team_id, away_team_id, bookmaker, ml_home, ml_away '
         'FROM odds_snapshots WHERE ml_home IS NOT NULL AND ml_away IS NOT NULL'),
    engine,
)
odds['game_date'] = pd.to_datetime(odds['game_date'])
print(f'odds rows: {len(odds):,}  books: {sorted(odds["bookmaker"].unique())}')

raw = load_team_games(engine)
outcomes = (
    raw[raw['is_home']][['game_id', 'won']]
    .rename(columns={'won': 'home_won'})
    .drop_duplicates(subset='game_id')
)
print(f'outcomes: {len(outcomes):,} games')

odds rows: 0  books: []
outcomes: 11,969 games


## 4. Five-strategy comparison

- **Flat $100**  — naive baseline.
- **Fixed 2 %**  — compound, not edge-aware.
- **Full Kelly** — log-utility-optimal under perfect knowledge; brutal variance.
- **0.25 × Kelly** — Thorp / MacLean-Ziemba's practitioner standard.
- **0.25 × Kelly + 2 % edge gate** — skip bets below a hard edge threshold.

All five share bankroll $10,000, min-edge 0.02 (backtest-level), side = best, 0.50 concurrent-exposure cap.

In [ ]:
strategies = {
    'flat_100':       FlatStake(unit=100.0),
    'fixed_2pct':     FixedFractional(pct=0.02),
    'full_kelly':     FullKelly(),
    'kelly_0.25':     FractionalKelly(multiplier=0.25),
    'kelly_0.25_gated': ThresholdKelly(min_edge=0.02, multiplier=0.25),
}

results = {}
for name, strat in strategies.items():
    cfg = BacktestConfig(
        strategy=strat,
        starting_bankroll=10_000,
        min_edge=0.02,
        side='best',
        bookmaker=None,
        max_concurrent_exposure=0.50,
    )
    r = run_backtest(preds, odds, outcomes, cfg)
    results[name] = r
    print(f'{name:20s}  {r.summary_pretty()}')

## 5. Summary metrics table

In [ ]:
rows = []
for name, r in results.items():
    s = r.summary
    rows.append({
        'strategy': name,
        'n_bets': s['n_bets'],
        'end_bankroll': s['ending_bankroll'],
        'ROI': s['roi'],
        'hit_rate': s['hit_rate'],
        'sharpe': s['sharpe'],
        'sortino': s['sortino'],
        'calmar': s['calmar'],
        'max_drawdown': s['max_drawdown'],
        'avg_edge': s['avg_edge'],
    })
summary_df = pd.DataFrame(rows)
summary_df.style.format({
    'end_bankroll': '${:,.0f}',
    'ROI': '{:+.2%}', 'hit_rate': '{:.2%}',
    'sharpe': '{:.2f}', 'sortino': '{:.2f}',
    'calmar': '{:.2f}', 'max_drawdown': '{:.2%}',
    'avg_edge': '{:+.3f}',
}).background_gradient(subset=['ROI', 'sharpe'], cmap='RdYlGn')

## 6. Equity curves

Full Kelly usually wins end-bankroll *when the model is correctly specified* — its volatility is the price. Fractional Kelly typically tracks close to it with a fraction of the drawdown.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for name, r in results.items():
    if r.equity.empty:
        continue
    ax.plot(pd.to_datetime(r.equity.index), r.equity.values, label=name, lw=1.5)
ax.axhline(10_000, color='gray', ls='--', lw=0.8, label='start ($10k)')
ax.set_xlabel('Date')
ax.set_ylabel('Bankroll ($)')
ax.set_title('Equity curves — 5 staking strategies\nbankroll on bet days only (no off-season fill)')
ax.legend(loc='upper left')
plt.tight_layout()

## 7. Drawdown curves

Drawdown is calculated against the **running peak** — `cummax()` — which correctly handles flat off-season periods in the sparse bet-day index.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for name, r in results.items():
    if r.equity.empty:
        continue
    dd = r.equity / r.equity.cummax() - 1
    ax.plot(pd.to_datetime(dd.index), dd.values, label=name, lw=1.2)
ax.set_xlabel('Date')
ax.set_ylabel('Drawdown')
ax.set_title('Drawdown curves — running peak (cummax)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend(loc='lower left')
plt.tight_layout()

## 8. Stake-size distribution + concurrent-Kelly audit

The cap is 50 % of bankroll across overlapping bets. On low-volume days the cap doesn't bind and Kelly fractions flow through as suggested. On high-volume days they get a proportional haircut. The check below should show Σfᵢ ≤ 0.50 on every game day, with ratios preserved.

In [ ]:
kelly_bets = results['kelly_0.25'].bets
if not kelly_bets.empty:
    daily_total = kelly_bets.groupby('game_date')['kelly_fraction_used'].sum()
    print(f'Days with ≥1 bet: {len(daily_total)}')
    print(f'Max daily Σf:     {daily_total.max():.4f}  (cap = 0.5000)')
    print(f'Days at cap:      {(daily_total > 0.499).sum()}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    axes[0].hist(kelly_bets['stake'], bins=40, color='steelblue', edgecolor='white')
    axes[0].set_title('Stake size distribution — 0.25× Kelly')
    axes[0].set_xlabel('Stake ($)')
    axes[0].set_ylabel('# bets')
    axes[1].hist(daily_total, bins=30, color='coral', edgecolor='white')
    axes[1].axvline(0.5, color='red', ls='--', lw=1.5, label='concurrent cap')
    axes[1].set_title('Σ(kelly_fraction_used) per game day')
    axes[1].set_xlabel('Day-total fraction')
    axes[1].legend()
    plt.tight_layout()

## 9. Bootstrap CIs on ROI and Sharpe

One or two seasons of bets is a noisy sample. Point estimates lie about significance. Bootstrap over the bet-level P&L distribution and report 95 % CIs.

In [ ]:
def bootstrap_metric(bets: pd.DataFrame, metric_fn, n_boot: int = 1000, seed: int = 0):
    """Resample bets with replacement; return (point, lo95, hi95)."""
    if bets.empty:
        return (float('nan'),) * 3
    rng = np.random.default_rng(seed)
    n = len(bets)
    samples = [metric_fn(bets.iloc[rng.integers(0, n, n)]) for _ in range(n_boot)]
    samples = np.array([s for s in samples if np.isfinite(s)])
    return (float(metric_fn(bets)), float(np.percentile(samples, 2.5)),
            float(np.percentile(samples, 97.5)))

def _roi(bets):
    st = bets['stake'].sum()
    return (bets['payout'].fillna(0).sum() / st) if st else 0.0

rows = []
for name, r in results.items():
    point, lo, hi = bootstrap_metric(r.bets, _roi)
    rows.append({'strategy': name, 'ROI': point, 'ROI_lo95': lo, 'ROI_hi95': hi})
ci_df = pd.DataFrame(rows)
ci_df.style.format({'ROI': '{:+.2%}', 'ROI_lo95': '{:+.2%}', 'ROI_hi95': '{:+.2%}'})

## 10. Recommendation & limits

**Which strategy to deploy?** Under log-utility and parameter uncertainty, the theoretical answer is fractional Kelly at some multiplier < 1 (Thorp; MacLean-Ziemba). `0.25 × Kelly + 2 % edge gate` is the version most defensible in a writeup: the gate filters low-confidence signals (analog to an alpha threshold in a factor portfolio); the 0.25× multiplier shrinks toward zero to protect against model misspecification; the 50 % concurrent cap is the gross-exposure analog of a fund-level risk budget.

**Limits of the analog.** Each bet has bounded downside (unlike leveraged positions). There's no cross-game correlation structure like the equity market has — but pace regimes and referee crews do introduce mild night-level correlation. Sample size is the binding constraint: even five full seasons of walk-forward bets filtered by 2 % edge is O(1,000) bets — small by factor-research standards.

**Known simplifications** (all documented in `docs/FINANCE.md`):
- Kaggle closes are one snapshot per game — no CLV (closing line value) analysis possible.
- Concurrency is defined at the game-*day* level, which treats a noon game and a 10 PM game as overlapping. Conservative; not broken.
- No transaction costs beyond vig (no withdrawal friction, no bet limits, no line shopping).

## 12. Meta-model: residual alpha test

Having established that the base model has no betting alpha vs. the closing line, the natural next quant-finance question is: **can a second-stage model extract residual alpha by using the market as a prior?**

Methodology (discipline enforced by `tests/test_meta_model.py`):

- **Side-level dataset.** Two rows per game — one for home side, one for away side. Symmetric learning; no home-bias artifact. Home-level compat shim (`side_to_home_preds`) for the existing backtest.
- **Market-only is the null hypothesis.** If meta does not beat `side_market_prob` out-of-sample on log-loss AND realized ROI, no alpha was found.
- **ROI + bootstrap 95 % CI, not a fixed win-rate threshold.** Moneylines vary so break-even varies per bet.
- **L2-regularized LR primary** (C=0.5, logit-space features). XGBoost (depth 2, heavy regularization) as a robustness check only.
- **Nested walk-forward** over 3 meta-test folds (2020-21, 2021-22, 2022-23 partial). OOS metrics only — no in-sample numbers count.

Details: [docs/FINANCE.md §8.1](../docs/FINANCE.md).

In [ ]:
from src.models.meta_model import (
    META_FEATURES_MINIMAL, build_side_frame, nested_walk_forward,
    variant_metrics_table, roi_per_fold, bootstrap_roi_ci, VARIANT_PROB_COLS,
)

side_df = build_side_frame(preds, odds)
print(f'Side-level rows: {len(side_df)} ({len(side_df) // 2} games × 2 sides)')
print(f'Folds: {sorted(side_df["fold"].unique())}')

for kind in ('lr', 'xgb'):
    side_df = nested_walk_forward(side_df, META_FEATURES_MINIMAL, model_kind=kind, initial_train_folds=1)

metrics_tbl = variant_metrics_table(side_df)
print('\nOut-of-sample metrics (same held-out rows across variants):')
print(metrics_tbl.round({'accuracy': 3, 'log_loss': 4, 'brier': 4, 'auc': 3, 'ece': 4}).to_string(index=False))

In [ ]:
MIN_EDGE = 0.02
print(f'Realized ROI at min_edge={MIN_EDGE:.2%}  (95% bootstrap CI, 2000 resamples at game level)')
print('-' * 70)
scope = side_df[side_df[['meta_prob_lr', 'meta_prob_xgb']].notna().any(axis=1)]
for name, col in VARIANT_PROB_COLS.items():
    if col not in scope.columns or scope[col].isna().all():
        continue
    point, lo, hi = bootstrap_roi_ci(scope, col, MIN_EDGE, n_boot=2000)
    n = ((scope[col].notna()) & ((scope[col] - scope['side_market_prob']) >= MIN_EDGE)).sum()
    print(f'  {name:10s}  n={n:<5d}  ROI={point:+.4f}  [{lo:+.4f}, {hi:+.4f}]')

print('\nmeta-LR ROI by fold:')
print(roi_per_fold(side_df, 'meta_prob_lr', MIN_EDGE).round(4).to_string(index=False))

### 12.1 Honest conclusion

**No residual alpha was found.** Measured on the same ~5,100 out-of-sample side-rows across 3 walk-forward test folds:

| Variant   | Log-loss  | ECE    | ROI @ 2% edge  | 95% CI            |
|-----------|----------:|-------:|---------------:|-------------------|
| base      | 0.642     | 0.012  | −2.95%         | [−8.74%, +3.29%]  |
| **market** (benchmark) | **0.610** | 0.010 | — (no bets)   | —                 |
| meta-LR   | 0.612     | 0.008  | −10.72%        | [−21.57%, +0.52%] |
| meta-XGB  | 0.624     | 0.040  | −5.31%         | [−12.28%, +1.80%] |

Reading the table against the pre-registered alpha criteria:

1. ❌ **ROI > 0** — no variant's aggregate ROI exceeded zero.
2. ❌ **Bootstrap CI excludes zero** — meta-LR's upper bound is +0.5% (just barely touches zero); meta-XGB's is +1.8%. Neither is significantly positive.
3. ❌ **Consistent sign across folds** — meta-LR is negative in both 2020-21 (−8.1%) and 2021-22 (−16.3%) (2022-23 had 0 bets due to sparse odds).
4. ❌ **Beats market on log-loss** — market-only wins (0.610 vs 0.612 for LR, 0.624 for XGB).

Running the full portfolio backtest (0.25× Kelly + 2% min-edge) confirms the finding at the P&L level: meta-LR drove the bankroll from \$10,000 to \$3,982 (−60% drawdown); meta-XGB to \$86 (−99%).

**Interpretation.** The base model carries accuracy but not alpha; the market prices in information the base model doesn't see; and combining the two does not manufacture alpha that wasn't there. This strengthens the §7 market-efficiency argument: even granting the model every advantage — including access to the market itself as a feature — residual edge does not emerge on the feature set this project has. Alpha would require additional information sources (line movement, intraday injury news, lineup announcements, crew-level referee data) that the base features cannot approximate.

This is the honest closure of the research loop.